# 2 — Buck Converter Controller Design

> **Goal.** Use the plant $G_{vd}(s)$ derived in notebook 1 to design a
> voltage-mode compensator, verify closed-loop stability and
> performance with analytical step responses, then validate the same
> controller in a Pulsim closed-loop simulation.

**Prerequisites**

- Notebook 1 (`01_buck_modeling.ipynb`).
- Basic loop-shaping vocabulary: gain crossover, phase margin,
  bandwidth.

**What you'll be able to do at the end**

1. State the closed-loop specifications (crossover, PM, DC error).
2. Read the phase margin and crossover of the *unmodified* plant
   $G_{vd}(s) \cdot k_{PWM}$ and explain why a buck without
   compensation oscillates.
3. Design a **type-II** compensator (PI + high-frequency pole) that
   meets the specs.
4. Verify the closed loop with both an analytical step and a Pulsim
   time-domain simulation.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from buck_model import (
    BuckParams, control_to_output_tf, line_to_output_tf,
    operating_point_report, linf_error, relative_rms_error,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

params = BuckParams()
print(operating_point_report(params))


## 1. Design specifications

For a digital point-of-load converter at $f_{sw} = 100$ kHz, a sensible
target is:

| Spec | Value | Rationale |
|---|---|---|
| Crossover frequency $f_c$ | $\approx f_{sw} / 10$ = 10 kHz | High enough for fast load transients, low enough to keep switching ripple out of the loop. |
| Phase margin (PM) | $\geq 45^\circ$ | Above this the closed loop has no resonant peaking; 60° gives ~5 % overshoot to a reference step. |
| DC steady-state error | 0 | Integrator → infinite DC loop gain. |
| Output ripple | $\leq 1$ % $V_o$ (120 mV) | Filter design, not directly a controller spec — set by L, C. |

The plant $G_{vd}(s)$ has a **double pole** at the LC corner
($f_n \approx 1.6$ kHz) with $Q \approx 2.4$ (lightly damped). At
$f_n$ the magnitude peaks at $Q \cdot V_g \approx 58$ dB and the phase
drops from 0° to −180°. If we just multiply by a gain to put crossover
at 10 kHz, the phase margin near 10 kHz is already past −180° → unstable.

The compensator's job: add a zero below the LC corner (pulls phase
back), keep an integrator (kills DC error), and add a pole above
crossover to roll off switching noise.


## 2. Modulator gain — what $k_{PWM}$ does to the loop

Between the controller's output `v_c` (a control voltage) and the duty
cycle commanded to the switch, there's a PWM modulator. If the
modulator uses a triangle / sawtooth ramp of amplitude $V_{ramp}$:

$$
d = \frac{v_c}{V_{ramp}} \;\implies\; k_{PWM} = \frac{1}{V_{ramp}}
$$

So if our controller output spans 0–5 V and the ramp is 5 V peak-peak,
$k_{PWM} = 1/5 = 0.2$. The complete loop gain is

$$
T(s) = G_c(s) \cdot k_{PWM} \cdot G_{vd}(s) \cdot H_{fb}
$$

with $H_{fb}$ the output sensing network (we'll use a unity divider for
simplicity — the math is the same with any other divider since it just
rescales the reference).


In [ ]:
Gvd = control_to_output_tf(params)
V_ramp = 5.0
k_pwm = 1.0 / V_ramp
H_fb = 1.0

# Plant + modulator (no compensator yet) — what we'd see if we closed
# the loop with a pure proportional controller of gain 1.
plant = signal.TransferFunction(
    np.array(Gvd.num) * k_pwm * H_fb,
    np.array(Gvd.den),
)
print(f"k_PWM = {k_pwm:.3f}  (V_ramp = {V_ramp} V)")
print(f"H_fb  = {H_fb:.3f}")


## 3. Uncompensated loop — why we need a compensator

Plot the Bode of the plant + modulator and read off:

- Where does $|T| = 0$ dB cross? (gain crossover frequency)
- What's the phase there? (phase margin = phase + 180°)


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2 * np.pi * f

_, mag, phase = signal.bode(plant, w=w)
crossover_idx = np.argmin(np.abs(mag))
f_cross_uncomp = f[crossover_idx]
pm_uncomp = 180 + phase[crossover_idx]

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_mag.semilogx(f, mag, label="Plant · $k_{PWM}$  (no compensator)")
ax_mag.axhline(0, color="k", linestyle=":", alpha=0.3)
ax_mag.axvline(f_cross_uncomp, color="r", linestyle=":", alpha=0.5,
               label=f"$f_c$ ≈ {f_cross_uncomp:.0f} Hz")
ax_ph.semilogx(f, phase, label="Plant · $k_{PWM}$  (no compensator)")
ax_ph.axhline(-180, color="k", linestyle=":", alpha=0.3)
ax_ph.axvline(f_cross_uncomp, color="r", linestyle=":", alpha=0.5)
ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.legend()
ax_ph.legend()
ax_mag.set_title("Uncompensated loop gain")
plt.tight_layout()
plt.show()

print(f"Crossover frequency = {f_cross_uncomp:8.1f} Hz")
print(f"Phase margin        = {pm_uncomp:8.2f} deg")
if pm_uncomp < 0:
    print("⚠️   Uncompensated loop is unstable — we'd have a sustained oscillation.")


## 4. Type-III compensator design (K-factor method)

The buck plant has a **lightly damped second-order resonance** at $f_n$
that drops the phase from 0° to −180° rapidly. Past $f_n$ the plant
phase is essentially −180°, leaving 0° of natural margin. To close the
loop at $f_c$ with a target PM of 60°, the **compensator** must
contribute the missing phase by itself.

We need a Type-III compensator (one integrator at the origin, **two**
zeros, **two** high-frequency poles). It can contribute up to +180°
of phase boost — enough to handle a buck plant with very high Q.

$$
G_c(s) = K \cdot \frac{(1 + s/\omega_z)^2}{s \cdot (1 + s/\omega_{p,1}) (1 + s/\omega_{p,2})}
$$

### The K-factor algorithm (Venable, 1983)

Given a target crossover $f_c$ and phase margin PM:

1. Measure the plant phase $\phi_{plant}$ at $f_c$.
2. Compute the **lead phase** the compensator must add at $f_c$
   (beyond the −90° from the integrator):

   $$
   \phi_{lead} = \text{PM} - 90° - \phi_{plant}
   $$

3. Split between the two zero-pole pairs: $\phi_{pair} = \phi_{lead} / 2$.

4. For each pair, the K-factor is

   $$
   K_{factor} = \tan^2\!\left( \frac{\phi_{pair}}{4} + 45° \right)
   $$

   placing zero at $\omega_z = \omega_c / \sqrt{K_{factor}}$ and pole at
   $\omega_p = \omega_c \cdot \sqrt{K_{factor}}$ — geometrically
   symmetric around the crossover, which maximizes the phase boost.

5. Scale the DC gain $K$ so $|T(j \omega_c)| = 1$ exactly.

The function below implements this in ~20 lines.


In [ ]:
def design_type3_kfactor(
    plant: signal.TransferFunction, f_c: float, pm_target: float
) -> tuple[signal.TransferFunction, float, float, float]:
    '''Design a Type-III compensator using the K-factor method.

    Returns (Gc, omega_z, omega_p, K_dc) where the same omega_z is used
    for both zeros and the same omega_p for both poles (symmetric
    K-factor placement around f_c).
    '''
    omega_c = 2 * np.pi * f_c
    _, _, ph_plant = signal.bode(plant, w=[omega_c])

    # Lead phase the compensator zeros must add beyond the integrator's -90°
    phi_lead = pm_target - 90.0 - ph_plant[0]
    # Clamp to a safe range — under 10° gives a near-degenerate design,
    # over 175° is unphysical (each zero/pole pair maxes at ~+90°).
    phi_lead = float(np.clip(phi_lead, 10.0, 175.0))
    phi_pair = phi_lead / 2

    k = np.tan(np.deg2rad(phi_pair / 2 + 45.0)) ** 2
    omega_z = omega_c / np.sqrt(k)
    omega_p = omega_c * np.sqrt(k)

    # Build the un-scaled compensator: (s + ω_z)² / [s · (s + ω_p)²]
    num0 = np.polymul([1.0, omega_z], [1.0, omega_z])
    den0 = np.polymul([1.0, 0.0], np.polymul([1.0, omega_p], [1.0, omega_p]))

    # Scale K so the loop gain magnitude is exactly 1 at f_c
    open0 = signal.TransferFunction(
        np.polymul(num0, plant.num),
        np.polymul(den0, plant.den),
    )
    _, mag0, _ = signal.bode(open0, w=[omega_c])
    K = 10.0 ** (-mag0[0] / 20)

    return signal.TransferFunction(K * num0, den0), omega_z, omega_p, float(K)


f_c_target = 5e3
pm_target = 60.0
Gc, omega_z, omega_p, K_dc = design_type3_kfactor(plant, f_c_target, pm_target)

print(f"Target:  f_c = {f_c_target/1e3:.1f} kHz, PM = {pm_target:.0f}°")
print()
print(f"Designed compensator (Type-III, K-factor):")
print(f"  zeros at   f_z = {omega_z/(2*np.pi):8.1f} Hz  (double)")
print(f"  HF poles at f_p = {omega_p/(2*np.pi):8.1f} Hz  (double)")
print(f"  DC gain K       = {K_dc:.4g}")


In [ ]:
# Loop gain = compensator · plant
T_open = signal.TransferFunction(
    np.polymul(Gc.num, plant.num),
    np.polymul(Gc.den, plant.den),
)

_, mag_T, ph_T = signal.bode(T_open, w=w)
idx_cross = np.argmin(np.abs(mag_T))
f_cross = f[idx_cross]
pm = 180 + ph_T[idx_cross]

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_mag.semilogx(f, mag, "C0--", alpha=0.5, label="Plant · $k_{PWM}$")
ax_mag.semilogx(f, mag_T, "C3", linewidth=2, label="Loop gain $T(s)$ (with compensator)")
ax_mag.axhline(0, color="k", linestyle=":", alpha=0.3)
ax_mag.axvline(f_cross, color="r", linestyle=":", alpha=0.5,
               label=f"$f_c$ ≈ {f_cross:.0f} Hz")
ax_ph.semilogx(f, phase, "C0--", alpha=0.5, label="Plant · $k_{PWM}$")
ax_ph.semilogx(f, ph_T, "C3", linewidth=2, label="Loop gain $T(s)$")
ax_ph.axhline(-180, color="k", linestyle=":", alpha=0.3)
ax_ph.axvline(f_cross, color="r", linestyle=":", alpha=0.5)
ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.legend(loc="best")
ax_ph.legend(loc="best")
ax_mag.set_title(f"Compensated loop: $f_c$ = {f_cross:.0f} Hz, PM = {pm:.1f}°")
plt.tight_layout()
plt.show()

print(f"Designed:  f_c = {f_c_target/1e3:.1f} kHz, PM target = {pm_target}°")
print(f"Achieved:  f_c = {f_cross/1e3:.2f} kHz, PM = {pm:.1f}°")


## 5. Closed-loop response

With $T(s)$ designed, the closed-loop transfer from reference
$v_{ref}$ to output is

$$
G_{cl}(s) = \frac{T(s)}{1 + T(s)}
$$

If the loop has enough crossover-to-pole margin, this should look like
a clean second-order response with the bandwidth set by $f_c$ and the
overshoot set by the phase margin (PM = 60° → ~5 % overshoot).


In [ ]:
# Closed loop = T / (1 + T)
num_T, den_T = T_open.num, T_open.den
num_cl = num_T
den_cl = np.polyadd(den_T, num_T)
G_cl = signal.TransferFunction(num_cl, den_cl)

# Reference step (e.g. ask for an extra 1 V at the output)
v_ref_step = 1.0
t = np.linspace(0, 5e-3, 5000)
_, y_cl = signal.step(G_cl, T=t)
v_o_cl = params.V_o + v_ref_step * y_cl

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t * 1e3, v_o_cl, label="Closed-loop response (analytical)")
ax.axhline(params.V_o + v_ref_step, color="g", linestyle=":", alpha=0.5,
           label=f"Target ({params.V_o + v_ref_step} V)")
ax.set_xlabel("Time [ms]")
ax.set_ylabel("$v_o$ [V]")
ax.set_title(f"Closed-loop step in $v_{{ref}}$ of +{v_ref_step} V around {params.V_o} V")
ax.legend()
plt.tight_layout()
plt.show()

# Rise time + overshoot
final = params.V_o + v_ref_step
rise_idx = np.argmax(v_o_cl >= final * 0.9)
overshoot = (np.max(v_o_cl) - final) / v_ref_step
settled_idx = np.where(np.abs(v_o_cl - final) > 0.02 * v_ref_step)[0]
settling = t[settled_idx[-1] if len(settled_idx) else 0] * 1e3

print(f"Rise time (0 → 90 %) = {t[rise_idx] * 1e3:6.3f} ms")
print(f"Overshoot            = {overshoot * 100:6.1f} %")
print(f"Settling (±2 %)      = {settling:6.3f} ms")


## 6. Discretization for digital implementation

A digital controller runs as a difference equation. The cleanest map
from continuous → discrete is the **Tustin (bilinear) transform**:

$$
s \leftarrow \frac{2}{T_s} \frac{1 - z^{-1}}{1 + z^{-1}}
$$

`scipy.signal.cont2discrete` does this for you. We pick the sample
period equal to the switching period — the most common choice for a
voltage-mode buck (update duty once per switching cycle).


In [ ]:
T_s = 1.0 / params.f_sw
# Pack as state-space first; cont2discrete handles tuples (num, den).
Gc_d_num, Gc_d_den, _ = signal.cont2discrete(
    (Gc.num, Gc.den), dt=T_s, method="bilinear"
)
print(f"Sample period T_s = {T_s * 1e6:.3f} µs")
print(f"Discrete-time numerator   coefficients: {Gc_d_num.flatten()}")
print(f"Discrete-time denominator coefficients: {Gc_d_den}")

# Difference-equation form: a_0 y[n] + a_1 y[n-1] + ... = b_0 u[n] + b_1 u[n-1] + ...
# For a digital PID or biquad implementation, normalize so a_0 = 1.
a = np.asarray(Gc_d_den) / Gc_d_den[0]
b = np.asarray(Gc_d_num).flatten() / Gc_d_den[0]
print()
print("Normalized recurrence (a[0] = 1):")
for i, bi in enumerate(b):
    print(f"  b[{i}] = {bi:+.6f}")
for i, ai in enumerate(a):
    print(f"  a[{i}] = {ai:+.6f}")
print()
print("Pseudo-code for the firmware (Direct-Form II Transposed):")
print("  err = v_ref - v_o_sensed")
print("  v_c = b[0]*err + state1")
print("  state1 = b[1]*err - a[1]*v_c + state2")
print("  state2 = b[2]*err - a[2]*v_c")
print("  duty = clamp(v_c / V_ramp, 0, 1)")


## 7. Pulsim closed-loop validation

To close the loop in Pulsim we use a `PIDController` virtual block (or a
custom signal block) sensing $v_o$ and emitting a duty signal into the
PWM source. For brevity here we use the in-tree `PIController` — a
type-II compensator boils down to a PI plus an HF pole, and the
in-tree block is sufficient for a sanity check.

If you don't have Pulsim built, this section will skip and the math
above still stands.


### 7.1 Switched-model simulation in pure Python

To **prove** the controller actually works on the switching waveform —
not just the small-signal average — we simulate the closed-loop buck
in pure Python using:

1. A forward-Euler integration of the switched buck model
   (ON / OFF state per PWM cycle).
2. The discretized compensator running once per switching period
   (sample-and-hold — the same way an MCU would update PWM duty in a
   digital power supply).
3. A reference voltage step at $t = t_{step}$ so we can watch the
   loop respond.

If the K-factor design from §4 is sound, we should see:
- The output **tracks** $V_{ref}$ at DC (integrator → zero steady-state error).
- The duty cycle **adjusts** from $D = V_{o}/V_{g}$ to the new value
  at the new operating point.
- The transient settles within roughly the analytical settling time
  from §5.
- Inductor current rebalances to $V_{ref}/R$.


In [ ]:
def simulate_closed_loop_buck(
    params,
    b: np.ndarray,
    a: np.ndarray,
    *,
    t_end: float = 5e-3,
    t_step: float = 1e-3,
    v_ref_initial: float = 12.0,
    v_ref_final: float = 13.0,
    V_ramp: float = 5.0,
    samples_per_period: int = 200,
):
    '''Forward-Euler switched-buck simulator with a digital compensator.

    States (continuous time, integrated at `dt_sim`):
        i_L  — inductor current  [A]
        v_o  — capacitor voltage [V]   (= output voltage)

    The compensator is a discrete-time Direct-Form II Transposed
    implementation of the Tustin-discretized type-III. It runs once per
    switching period (sample-and-hold) — same way an MCU's PWM ISR
    would update duty in firmware.

    Returns a dict of arrays for plotting.
    '''
    T_s = 1.0 / params.f_sw
    dt_sim = T_s / samples_per_period
    n_steps = int(t_end / dt_sim) + 1

    # Plant state
    i_L = 0.0
    v_o = 0.0

    # Compensator state — n states for an n-th order DF-II Transposed
    n_state = len(a) - 1
    state = np.zeros(n_state)
    duty = 0.5

    # Recording arrays — downsampled so plots aren't 100k points each
    record_every = max(1, samples_per_period // 50)
    n_rec = n_steps // record_every + 1
    t_hist = np.zeros(n_rec)
    v_o_hist = np.zeros(n_rec)
    i_L_hist = np.zeros(n_rec)
    duty_hist = np.zeros(n_rec)
    v_ref_hist = np.zeros(n_rec)
    rec_idx = 0

    for i in range(n_steps):
        t = i * dt_sim
        v_ref = v_ref_initial if t < t_step else v_ref_final

        # Compensator update — once per switching period
        cycle_pos_int = i % samples_per_period
        if cycle_pos_int == 0:
            err = v_ref - v_o
            # Direct-Form II Transposed update
            v_c = b[0] * err + state[0]
            new_state = np.zeros_like(state)
            for j in range(n_state - 1):
                new_state[j] = b[j + 1] * err - a[j + 1] * v_c + state[j + 1]
            new_state[n_state - 1] = b[n_state] * err - a[n_state] * v_c
            state = new_state
            duty = float(np.clip(v_c / V_ramp, 0.0, 1.0))

        # PWM: high during first `duty` fraction of each period
        switch_on = (cycle_pos_int / samples_per_period) < duty

        # Switched buck ODE (forward Euler)
        v_L = (params.V_g - v_o) if switch_on else (-v_o)   # freewheel via diode
        i_C = i_L - v_o / params.R
        i_L += (v_L / params.L) * dt_sim
        i_L = max(i_L, 0.0)   # ideal diode prevents reverse current → CCM/DCM transition
        v_o += (i_C / params.C) * dt_sim

        # Record
        if i % record_every == 0 and rec_idx < n_rec:
            t_hist[rec_idx] = t
            v_o_hist[rec_idx] = v_o
            i_L_hist[rec_idx] = i_L
            duty_hist[rec_idx] = duty
            v_ref_hist[rec_idx] = v_ref
            rec_idx += 1

    return {
        "t":     t_hist[:rec_idx],
        "v_o":   v_o_hist[:rec_idx],
        "i_L":   i_L_hist[:rec_idx],
        "duty":  duty_hist[:rec_idx],
        "v_ref": v_ref_hist[:rec_idx],
    }


In [ ]:
# Run the closed-loop simulation: settle for 1 ms at V_ref = 12 V,
# then step to V_ref = 13 V at t = 1 ms.
sim = simulate_closed_loop_buck(
    params,
    b=b,    # discretized numerator (from cell 16)
    a=a,    # discretized denominator (from cell 16)
    t_end=5e-3,
    t_step=1e-3,
    v_ref_initial=12.0,
    v_ref_final=13.0,
    V_ramp=V_ramp,
)

print(f"Simulated {len(sim['t'])} recorded samples over {sim['t'][-1] * 1e3:.2f} ms")
print(f"Pre-step settled V_o (mean 0.8-1.0 ms): "
      f"{np.mean(sim['v_o'][(sim['t'] > 0.8e-3) & (sim['t'] < 1.0e-3)]):.4f} V "
      f"(target = 12.0 V)")
print(f"Post-step settled V_o (mean 4.8-5.0 ms): "
      f"{np.mean(sim['v_o'][sim['t'] > 4.8e-3]):.4f} V "
      f"(target = 13.0 V)")
print(f"Pre-step duty:  {np.mean(sim['duty'][(sim['t'] > 0.8e-3) & (sim['t'] < 1.0e-3)]):.4f}  "
      f"(expect D₁ = 12/24 = 0.500)")
print(f"Post-step duty: {np.mean(sim['duty'][sim['t'] > 4.8e-3]):.4f}  "
      f"(expect D₂ = 13/24 = 0.542)")


In [ ]:
# Plot the four key waveforms.
fig, axs = plt.subplots(4, 1, figsize=(11, 11), sharex=True)

axs[0].plot(sim['t'] * 1e3, sim['v_o'], 'C0', linewidth=1.0, label="$v_o$ (switched)")
axs[0].plot(sim['t'] * 1e3, sim['v_ref'], 'C3--', linewidth=2.0, label="$v_{ref}$")
axs[0].axvline(1.0, color='k', linestyle=':', alpha=0.4, label="step")
axs[0].set_ylabel("Output voltage [V]")
axs[0].set_title("Closed-loop buck: step in $v_{ref}$ from 12 V → 13 V at $t$ = 1 ms")
axs[0].legend(loc="lower right")

axs[1].plot(sim['t'] * 1e3, sim['i_L'], 'C1', linewidth=1.0)
axs[1].axvline(1.0, color='k', linestyle=':', alpha=0.4)
axs[1].set_ylabel("Inductor current [A]")
axs[1].axhline(12.0 / params.R, color='k', linestyle=':', alpha=0.3,
               label=f"pre-step $V_{{ref}}/R$ = {12.0 / params.R:.2f} A")
axs[1].axhline(13.0 / params.R, color='r', linestyle=':', alpha=0.3,
               label=f"post-step $V_{{ref}}/R$ = {13.0 / params.R:.2f} A")
axs[1].legend(loc="lower right")

axs[2].plot(sim['t'] * 1e3, sim['duty'], 'C2', linewidth=1.2)
axs[2].axvline(1.0, color='k', linestyle=':', alpha=0.4)
axs[2].axhline(0.5, color='k', linestyle=':', alpha=0.3,
               label="pre-step D = 0.500")
axs[2].axhline(13.0 / params.V_g, color='r', linestyle=':', alpha=0.3,
               label=f"post-step D = {13.0 / params.V_g:.3f}")
axs[2].set_ylabel("Duty cycle")
axs[2].legend(loc="lower right")

# Tracking error
axs[3].plot(sim['t'] * 1e3, sim['v_ref'] - sim['v_o'], 'C4', linewidth=1.0)
axs[3].axvline(1.0, color='k', linestyle=':', alpha=0.4)
axs[3].axhline(0, color='k', linestyle=':', alpha=0.3)
axs[3].set_ylabel("Tracking error\n$v_{ref} - v_o$ [V]")
axs[3].set_xlabel("Time [ms]")

plt.tight_layout()
plt.show()


In [ ]:
# Closed-loop performance metrics on the step response
mask_after = sim['t'] > 1.0e-3
t_after = sim['t'][mask_after] - 1.0e-3
v_o_after = sim['v_o'][mask_after]

# Peak overshoot above the new reference
v_o_max = np.max(v_o_after)
overshoot_pct = (v_o_max - 13.0) / (13.0 - 12.0) * 100
# Settling time: when |v_o - v_ref| stays < 2 % of step magnitude
settled = np.abs(v_o_after - 13.0) < 0.02 * (13.0 - 12.0)
settled_continuous = np.where(~settled)[0]
settling_idx = settled_continuous[-1] + 1 if len(settled_continuous) > 0 else 0
settling_ms = t_after[min(settling_idx, len(t_after) - 1)] * 1e3
# Rise time: 10 % to 90 %
v_o_10pct = 12.0 + 0.1 * (13.0 - 12.0)
v_o_90pct = 12.0 + 0.9 * (13.0 - 12.0)
rise_start_idx = np.argmax(v_o_after >= v_o_10pct)
rise_end_idx = np.argmax(v_o_after >= v_o_90pct)
rise_time_ms = (t_after[rise_end_idx] - t_after[rise_start_idx]) * 1e3

print("Closed-loop step-response metrics (V_ref: 12 V → 13 V)")
print(f"  Rise time  (10% → 90%)   = {rise_time_ms:6.3f} ms")
print(f"  Peak overshoot           = {overshoot_pct:6.2f} %")
print(f"  Settling time (±2 %)     = {settling_ms:6.3f} ms")
print()

# Steady-state regulation: zero error?
ss_error = 13.0 - np.mean(sim['v_o'][sim['t'] > 4.5e-3])
print(f"  Steady-state error       = {ss_error * 1e3:+7.2f} mV ({ss_error / 13.0 * 100:+.3f} %)")
print()
if abs(ss_error) < 0.01 and overshoot_pct < 30 and settling_ms < 2.0:
    print("✅  Closed-loop controller PROVEN: tracks reference with < 10 mV DC error,")
    print(f"    overshoot {overshoot_pct:.1f} %, settles in {settling_ms:.2f} ms.")
else:
    print("⚠️   Closed-loop response off-target — revisit f_c or PM in section 4.")


## 8. Summary

You took the small-signal plant $G_{vd}(s)$ derived in notebook 1,
sized a type-II compensator that lifts the gain at low frequency,
zeros out the LC peak, and rolls off above $f_c$. The closed-loop
analytical step response shows the targeted rise time and overshoot.
A digital-form recurrence is ready to drop into firmware.

**Suggested exercises**

1. Re-tune for $f_c = 5$ kHz with PM = 75°. How much does the overshoot
   improve? How much does the settling time stretch?
2. Add a 50 % load step ($R$: 2.4 → 1.2 Ω) at $t = 2$ ms in Pulsim and
   measure the recovery time. Compare with the analytical
   $Z_{out}^{closed}(s) = Z_{out}(s) / (1 + T(s))$ prediction.
3. Replace the type-II with a **type-III** (add a second zero/pole
   pair). Where does the extra phase boost let you push $f_c$?
